**to import data**

In [ ]:
import os 
for d, _, files in os.walk('/kaggle/input'):
    for name in files :
        print(d + '/' + name)

**to import pandas and analysis data **

In [ ]:
import pandas as pd
#df is Variable
df= pd.read_csv('/kaggle/input/car-price-01/Car Price.csv')
df.head()

**to Chek how much Row & Coumn **

In [ ]:
df.shape

**to see the tayp of data any Coulmn**

In [ ]:
#this is important to know what Encoding requires 
df.dtypes

**we identify the columns with missing Values to see what we will do **

In [ ]:
#.isna see null V 
#.mean see average
df.isna().mean().sort_values(ascending=False).head(26)


In [ ]:
#stor messing ratios in variable 
null_ratios=df.isna().mean()
null_ratios

In [ ]:
target='price'
#to see if price C or not
df[target].head()

** now import matplotlib **

In [ ]:
import matplotlib.pyplot as plt
df[target].hist(bins=30)
plt.title("Distribution of car prices ")
plt.xlabel('price')
plt.ylabel('count car')
plt.show()

In [ ]:
df['enginesize'].hist(bins=30)
plt.title("Distribution of engine size")
plt.xlabel('enginesize')
plt.ylabel('cunt')
plt.show()
#Most engines are between 80 and 160 

** for horse power **

In [ ]:
df['horsepower'].hist(bins=40)
plt.title('Distribution of horsepower')
plt.xlabel('horsepower')
plt.ylabel('count')
plt.show()

**The relationship between engine size and price**

In [ ]:
plt.scatter(df['enginesize'],df[target])
plt.title('enginesize vs price')
plt.xlabel('enginesize')
plt.ylabel('price')
plt.show()

**The relationship between horsepower and price**

In [ ]:
plt.scatter(df['horsepower'], df[target])
plt.title('horsepower vs price')
plt.xlabel('horsepower')
plt.ylabel('price')
plt.show()

**The relationship between Curbweight and price**

In [ ]:
plt.scatter(df['curbweight'],df[target])
plt.title('curbweight vs price ')
plt.xlabel('curbweight')
plt.ylabel('price')
plt.show()

**We now see all the relationships between the columns**

In [ ]:
num_cols = df.select_dtypes(include=[float,int]).columns
num_cols

corr=df[num_cols].corr()
corr

**Drwing Heatmap**

In [ ]:
plt.imshow(corr, cmap='coolwarm', interpolation='nearest')
plt.colorbar(label='Correlation')
plt.xticks(range(len(num_cols)), num_cols, rotation=90)
plt.yticks(range(len(num_cols)), num_cols)
plt.title('Correlation Heatmap (numeric features)')
plt.tight_layout()
plt.show()

**Numbers**

In [ ]:
corr_with_price = corr['price'].sort_values(ascending=False)
corr_with_price

**We begin with a detailed enumeration of three features.**

In [ ]:
cols_for_stats = ['price', 'enginesize', 'horsepower']
df[cols_for_stats].describe()

**We calculate Q1, Q3, IQR, Upper, Lower for the price**

In [ ]:
col = 'price'

q1 = df[col].quantile(0.25)
q3 = df[col].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print(' q1 =',q1,' q3 =', q3,' iqr =', iqr,' lower =', lower,' upper =', upper)

**We calculate Q1, Q3, IQR, Upper, Lower for the enginesize**

In [ ]:
col = 'enginesize'

q1 = df[col].quantile(0.25)
q3 = df[col].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print(' q1 =',q1,' q3 =', q3,' iqr =', iqr,' lower =', lower,' upper =', upper)

**We calculate Q1, Q3, IQR, Upper, Lower for the horsepower**

In [ ]:
col = 'horsepower'

q1 = df[col].quantile(0.25)
q3 = df[col].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print(' q1 =',q1,' q3 =', q3,' iqr =', iqr,' lower =', lower,' upper =', upper)

**We only select classes whose prices are within normal limits.**

In [ ]:
col = 'price'

q1 = df[col].quantile(0.25)
q3 = df[col].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

df_clean = df[(df[col] >= lower) & (df[col] <= upper)]
df_clean.shape

**We are now working on clean data.**

In [ ]:
data = df_clean   

y = data['price']
X = data.drop(columns=['price'])

X.head()

In [ ]:
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
numeric_features

In [ ]:
categorical_features = X.select_dtypes(exclude=['int64', 'float64']).columns.tolist()
categorical_features

**Data segmentation: Train / Test**

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 80% Train  and 20% Test
    random_state=42     # A fixed number so that the division is repeated in the same way
)
X_train.shape, X_test.shape


** import wgat we need for modling**

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import math

**We are building Transformer for digital columns**

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
numeric_transformer

In [ ]:
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
categorical_transformer

**We combine the two in ColumnTransformer**

In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)
preprocess

**Model definition + Pipeline**

In [ ]:
model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

model_pipe = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', model)
])

model_pipe

**We train the model (fit)**

In [ ]:
model_pipe.fit(X_train, y_train)

**We measure the model's performance on the Training set**

In [ ]:
train_preds = model_pipe.predict(X_train)

from sklearn.metrics import mean_squared_error, r2_score
import math

train_rmse = math.sqrt(mean_squared_error(y_train, train_preds))
train_r2 = r2_score(y_train, train_preds)

train_rmse, train_r2

**we will test modl**

In [ ]:
test_preds = model_pipe.predict(X_test)

test_rmse = math.sqrt(mean_squared_error(y_test, test_preds))
test_r2 = r2_score(y_test, test_preds)

test_rmse, test_r2

In [ ]:
results_df = pd.DataFrame(
    {
        "RMSE": [train_rmse, test_rmse],
        "R²":   [train_r2,   test_r2]
    },
    index=["Train", "Test"]
)

results_df.style.format({
    "RMSE": "{:,.2f}",  
    "R²":   "{:.3f}" 
})